# Avance 2 — Modelo Relacional de Ventas

Este notebook toma el archivo `ventasTransformed.csv` (exportado desde Power Query) y construye las tablas relacionales necesarias:
- Ciudades, Sucursales, Vendedores, Clientes, Productos, Facturas, DetalleFacturas.

Al final se genera `modeloVentas.xlsx` con todas las hojas listas para importar en Power BI.


Importar librerías y leer el CSV

In [34]:
import pandas as pd
import numpy as np

# Leer todo como texto al inicio para evitar problemas de inferencia automática
df_raw = pd.read_csv("ventasTransformed.csv", dtype=str).fillna("")

# Normalizar nombres de columnas (quitar espacios al final/inicio)
df_raw.columns = [c.strip() for c in df_raw.columns]

print("Filas leídas:", len(df_raw))
df_raw.head()


Filas leídas: 30000


,VentaID,FechaVenta,HoraVenta,SucursalNombre,VendedorNombre,Cliente_Nombre,GeneroCliente,edadcliente,EmailCliente,TelefonoCliente,...,NombreProducto3,MarcaProducto3,CantidadProducto3,PrecioUnitarioProducto3,SubtotalProducto3,DescuentoVenta,TotalVenta,MesVenta,AnoVenta,CiudadSucursal
0,1,"Thursday, December 31, 2015",5:42:43 AM,TechCore Pereira,Amílcar Ortega-Alberto,Bienvenida Nebot-Fiol,F,37,bienvenida19@hotmail.com,34806548767,...,Dell Latitude 7420,Dell,1,5600000,5600000,0,31200000,December,2015,Pereira
1,2,"Saturday, March 23, 2019",7:03:21 PM,TechCore Medellín #1,Ana Sofía Llopis Blázquez,Teófila Bueno-Novoa,F,43,teófila35@gmail.com,34947255990,...,No Aplica,No Aplica,0,0,0,0,10800000,March,2019,Medellín
2,3,"Sunday, December 23, 2018",2:32:34 AM,TechCore Medellín #2,Juan José Porcel-Riera,Gilberto Chamorro Catalá,M,38,gilberto57@hotmail.com,34978810249,...,HP Pavilion 15,HP,1,3500000,3500000,0,21900000,December,2018,Medellín
3,4,"Saturday, November 14, 2015",12:37:36 AM,TechCore Medellín #2,Juan José Porcel-Riera,Máximo Coronado Huerta,M,30,máximo65@hotmail.com,34825429634,...,No Aplica,No Aplica,0,0,0,0,13600000,November,2015,Medellín
4,5,"Tuesday, November 29, 2016",10:34:20 AM,TechCore Cali,Jacinta Juárez Marín,Yago Oliver,M,51,yago38@yahoo.com,34988285275,...,Microsoft Surface Laptop 4,Microsoft,1,5200000,5200000,0,13900000,November,2016,Cali


### Clientes
Creamos una tabla de clientes única.  
Convertimos teléfonos a texto y dejamos "No Especificado" cuando no hay dato.


In [35]:
clientes_tmp = df_raw[[
    "Cliente_Nombre", "GeneroCliente", "edadcliente",
    "EmailCliente", "TelefonoCliente", "Direccion_Cliente"
]].copy()

# Normalizaciones simples
clientes_tmp["Cliente_Nombre"] = clientes_tmp["Cliente_Nombre"].str.strip().str.title().replace("", "No Especificado")
clientes_tmp["GeneroCliente"] = clientes_tmp["GeneroCliente"].str.strip().replace("", pd.NA)
clientes_tmp["edadcliente"] = pd.to_numeric(clientes_tmp["edadcliente"], errors="coerce").astype("Int64")
clientes_tmp["EmailCliente"] = clientes_tmp["EmailCliente"].str.strip().str.lower().replace("", pd.NA)
clientes_tmp["TelefonoCliente"] = clientes_tmp["TelefonoCliente"].str.strip().replace("", "No Especificado")
clientes_tmp["Direccion_Cliente"] = clientes_tmp["Direccion_Cliente"].str.strip().replace("", "No Especificado")

# Clave de deduplicación: usar email cuando esté, si no usar nombre+telefono
clientes_tmp["dedup_key"] = clientes_tmp["EmailCliente"].where(clientes_tmp["EmailCliente"].notna(),
                                                               clientes_tmp["Cliente_Nombre"] + "|" + clientes_tmp["TelefonoCliente"])

clientes_unique = clientes_tmp.drop_duplicates(subset=["dedup_key"]).reset_index(drop=True)

clientes_df = pd.DataFrame({
    "ClienteID": range(1, len(clientes_unique)+1),
    "Nombre": clientes_unique["Cliente_Nombre"],
    "Genero": clientes_unique["GeneroCliente"],
    "Edad": clientes_unique["edadcliente"],
    "Telefono": clientes_unique["TelefonoCliente"],
    "Email": clientes_unique["EmailCliente"].fillna("No Especificado"),
    "Direccion": clientes_unique["Direccion_Cliente"]
})

print("Clientes únicos:", len(clientes_df))
clientes_df.head()


Clientes únicos: 15816


,ClienteID,Nombre,Genero,Edad,Telefono,Email,Direccion
0,1,Bienvenida Nebot-Fiol,F,37,34806548767,bienvenida19@hotmail.com,cll 52 #32-98
1,2,Teófila Bueno-Novoa,F,43,34947255990,teófila35@gmail.com,cll 96 #81-94
2,3,Gilberto Chamorro Catalá,M,38,34978810249,gilberto57@hotmail.com,cra 28 #79-85
3,4,Máximo Coronado Huerta,M,30,34825429634,máximo65@hotmail.com,cll 5 #89-26
4,5,Yago Oliver,M,51,34988285275,yago38@yahoo.com,cra 32 #97-86


### Ciudades
Sacamos las ciudades desde la columna `CiudadSucursal`.  
Si falta, queda "No Especificado".


In [36]:
ciudades = df_raw["CiudadSucursal"].astype(str).str.strip().str.title().replace("", "No Especificado")
ciudades_df = ciudades.drop_duplicates().reset_index(drop=True).to_frame(name="NombreCiudad")
ciudades_df.insert(0, "CiudadID", range(1, len(ciudades_df)+1))

print("Ciudades:", len(ciudades_df))
ciudades_df.head()


Ciudades: 4


,CiudadID,NombreCiudad
0,1,Pereira
1,2,Medellín
2,3,Cali
3,4,Bogotá


### Sucursales
Cada sucursal tiene un SucursalID y referencia a CiudadID.


In [37]:
suc_tmp = df_raw[["SucursalNombre", "CiudadSucursal"]].copy()
suc_tmp["SucursalNombre"] = suc_tmp["SucursalNombre"].str.strip()
suc_tmp["CiudadSucursal"] = suc_tmp["CiudadSucursal"].astype(str).str.strip().str.title().replace("", "No Especificado")

sucursales_df = suc_tmp.drop_duplicates(subset=["SucursalNombre"]).reset_index(drop=True)
sucursales_df = sucursales_df.rename(columns={"CiudadSucursal": "NombreCiudad"})
sucursales_df = sucursales_df.merge(ciudades_df, on="NombreCiudad", how="left")

sucursales_df.insert(0, "SucursalID", range(1, len(sucursales_df)+1))
sucursales_df = sucursales_df[["SucursalID", "SucursalNombre", "CiudadID"]]

print("Sucursales:", len(sucursales_df))
sucursales_df.head()


Sucursales: 6


,SucursalID,SucursalNombre,CiudadID
0,1,TechCore Pereira,1
1,2,TechCore Medellín #1,2
2,3,TechCore Medellín #2,2
3,4,TechCore Cali,3
4,5,TechCore Bogotá #2,4


### Vendedores
Generamos un listado único de vendedores con su ID.


In [38]:
vendedores = df_raw["VendedorNombre"].astype(str).str.strip().str.title().replace("", "No Especificado")
vendedores_df = vendedores.drop_duplicates().reset_index(drop=True).to_frame(name="VendedorNombre")
vendedores_df.insert(0, "VendedorID", range(1, len(vendedores_df)+1))

print("Vendedores:", len(vendedores_df))
vendedores_df.head()


Vendedores: 30


,VendedorID,VendedorNombre
0,1,Amílcar Ortega-Alberto
1,2,Ana Sofía Llopis Blázquez
2,3,Juan José Porcel-Riera
3,4,Jacinta Juárez Marín
4,5,Clímaco Salinas Zabaleta


### Productos
Unimos las tres columnas de producto en una sola lista, limpiamos y hacemos un catálogo único.
El precio del producto se toma como la mediana de los precios observados (si hay varios).


In [39]:
prod_frames = []
for i in [1,2,3]:
    cols = [f"NombreProducto{i}", f"MarcaProducto{i}", f"PrecioUnitarioProducto{i}"]
    tmp = df_raw[cols].rename(columns={
        cols[0]:"NombreProducto", cols[1]:"MarcaProducto", cols[2]:"PrecioUnitario"
    }).copy()
    tmp["NombreProducto"] = tmp["NombreProducto"].astype(str).str.strip()
    tmp["MarcaProducto"] = tmp["MarcaProducto"].astype(str).str.strip()
    tmp["PrecioUnitario"] = pd.to_numeric(tmp["PrecioUnitario"], errors="coerce")
    prod_frames.append(tmp)

productos_all = pd.concat(prod_frames, ignore_index=True)
productos_all = productos_all[productos_all["NombreProducto"].str.strip() != ""]  # quitar vacíos

# Calcular precio de referencia (mediana) por nombre+marca
precio_ref = (productos_all
              .groupby(["NombreProducto","MarcaProducto"], dropna=False)["PrecioUnitario"]
              .median()
              .reset_index()
              .rename(columns={"PrecioUnitario":"PrecioUnitarioMediana"}))

productos_unique = productos_all.drop_duplicates(subset=["NombreProducto","MarcaProducto"]).merge(precio_ref, on=["NombreProducto","MarcaProducto"], how="left")
productos_unique = productos_unique.reset_index(drop=True)
productos_unique.insert(0, "ProductoID", range(1, len(productos_unique)+1))

productos_df = productos_unique[["ProductoID","NombreProducto","MarcaProducto","PrecioUnitarioMediana"]].rename(columns={"PrecioUnitarioMediana":"PrecioUnitario"})

print("Productos únicos:", len(productos_df))
productos_df.head()


Productos únicos: 41


,ProductoID,NombreProducto,MarcaProducto,PrecioUnitario
0,1,Apple MacBook Pro 16,Apple,9600000.0
1,2,Acer Nitro 5,Acer,4000000.0
2,3,Dell Latitude 7420,Dell,5600000.0
3,4,Asus TUF Gaming A15,Asus,4800000.0
4,5,HP Spectre x360,HP,5200000.0


### Facturas
Usamos la columna `VentaID` como `FacturaID` para mantener trazabilidad.  
Aseguramos que FacturaID sea único y mapeamos las claves foráneas.


In [40]:
# Tomar columnas de interés y usar VentaID como FacturaID
fact_temp = df_raw[[
    "VentaID","FechaVenta","HoraVenta","SucursalNombre","VendedorNombre",
    "Cliente_Nombre","MetodoPago","DescuentoVenta","TotalVenta","MesVenta","AnoVenta"
]].copy()

# Normalizar
fact_temp["VentaID"] = fact_temp["VentaID"].astype(str).str.strip()
fact_temp = fact_temp.drop_duplicates(subset=["VentaID"]).reset_index(drop=True)  # asegurar una fila por VentaID

# Usar VentaID como FacturaID (convertir a entero si aplica)
# Si VentaID no es numérico, lo dejamos como texto; para Power BI conviene usar entero cuando sea posible.
def safe_int(x):
    try:
        return int(float(x))
    except:
        return x

fact_temp["FacturaID"] = fact_temp["VentaID"].apply(safe_int)

# Mapear SucursalID
fact_temp["SucursalNombre"] = fact_temp["SucursalNombre"].astype(str).str.strip()
fact_temp = fact_temp.merge(sucursales_df[["SucursalID","SucursalNombre"]], on="SucursalNombre", how="left")

# Mapear VendedorID
fact_temp["VendedorNombre"] = fact_temp["VendedorNombre"].astype(str).str.strip().str.title()
fact_temp = fact_temp.merge(vendedores_df[["VendedorID","VendedorNombre"]], on="VendedorNombre", how="left")

# Mapear ClienteID por nombre (usamos Nombre en clientes_df)
fact_temp["Cliente_Nombre"] = fact_temp["Cliente_Nombre"].astype(str).str.strip().str.title()
fact_temp = fact_temp.merge(clientes_df[["ClienteID","Nombre"]], left_on="Cliente_Nombre", right_on="Nombre", how="left")

# Seleccionar y renombrar columnas finales
facturas = fact_temp[[
    "FacturaID","VentaID","FechaVenta","HoraVenta","SucursalID","ClienteID","VendedorID",
    "MetodoPago","DescuentoVenta","TotalVenta","MesVenta","AnoVenta"
]].copy()

# Asegurar tipos numéricos donde corresponda
facturas["FacturaID"] = pd.to_numeric(facturas["FacturaID"], errors="coerce").astype("Int64")
facturas["SucursalID"] = facturas["SucursalID"].astype("Int64")
facturas["ClienteID"] = facturas["ClienteID"].astype("Int64")
facturas["VendedorID"] = facturas["VendedorID"].astype("Int64")
facturas["DescuentoVenta"] = pd.to_numeric(facturas["DescuentoVenta"], errors="coerce").fillna(0.0).astype(float)
facturas["TotalVenta"] = pd.to_numeric(facturas["TotalVenta"], errors="coerce").astype(float)

print("Facturas creadas:", len(facturas))
facturas.head()


Facturas creadas: 30013


,FacturaID,VentaID,FechaVenta,HoraVenta,SucursalID,ClienteID,VendedorID,MetodoPago,DescuentoVenta,TotalVenta,MesVenta,AnoVenta
0,1,1,"Thursday, December 31, 2015",5:42:43 AM,1,1,1,Tarjeta Crédito,0.0,31200000.0,December,2015
1,2,2,"Saturday, March 23, 2019",7:03:21 PM,2,2,2,Billetera Digital,0.0,10800000.0,March,2019
2,3,3,"Sunday, December 23, 2018",2:32:34 AM,3,3,3,Tarjeta Débito,0.0,21900000.0,December,2018
3,4,4,"Saturday, November 14, 2015",12:37:36 AM,3,4,3,Billetera Digital,0.0,13600000.0,November,2015
4,5,5,"Tuesday, November 29, 2016",10:34:20 AM,4,5,4,Tarjeta Crédito,0.0,13900000.0,November,2016


### DetalleFacturas
Convertimos las columnas de producto (1..3) a formato largo y asociamos ProductoID y FacturaID.  
Calculamos el descuento por línea (porcentaje de la factura) y el SubtotalNeto.


In [41]:
# Convertir a formato largo
detalle_frames = []
for i in [1,2,3]:
    cols = [
        "VentaID",
        f"NombreProducto{i}", f"MarcaProducto{i}",
        f"CantidadProducto{i}", f"PrecioUnitarioProducto{i}", f"SubtotalProducto{i}"
    ]
    dfp = df_raw[cols].rename(columns={
        f"NombreProducto{i}":"NombreProducto",
        f"MarcaProducto{i}":"MarcaProducto",
        f"CantidadProducto{i}":"Cantidad",
        f"PrecioUnitarioProducto{i}":"PrecioUnitario",
        f"SubtotalProducto{i}":"Subtotal",
        "VentaID":"VentaID"
    }).copy()
    # limpiar
    dfp["NombreProducto"] = dfp["NombreProducto"].astype(str).str.strip()
    dfp = dfp[dfp["NombreProducto"] != ""]  # sólo filas con producto
    dfp["Cantidad"] = pd.to_numeric(dfp["Cantidad"], errors="coerce").fillna(0).astype(int)
    dfp["PrecioUnitario"] = pd.to_numeric(dfp["PrecioUnitario"], errors="coerce").astype(float)
    dfp["Subtotal"] = pd.to_numeric(dfp["Subtotal"], errors="coerce").astype(float)
    detalle_frames.append(dfp)

detalle_all = pd.concat(detalle_frames, ignore_index=True)

# Mapear ProductoID por nombre+marca
detalle_all = detalle_all.merge(productos_df[["ProductoID","NombreProducto","MarcaProducto"]],
                                on=["NombreProducto","MarcaProducto"], how="left")

# Mapear FacturaID desde VentaID usando la tabla facturas
# Aseguramos tipo VentaID similar
detalle_all["VentaID"] = detalle_all["VentaID"].astype(str).str.strip()
map_v = facturas[["VentaID","FacturaID"]].drop_duplicates()
detalle_all = detalle_all.merge(map_v, on="VentaID", how="left")

# Asegurar FacturaID tipo Int64
detalle_all["FacturaID"] = pd.to_numeric(detalle_all["FacturaID"], errors="coerce").astype("Int64")

# Añadir DescuentoVenta desde facturas para calcular Descuento por línea
detalle_all = detalle_all.merge(facturas[["FacturaID","DescuentoVenta"]], on="FacturaID", how="left")
detalle_all["DescuentoVenta"] = detalle_all["DescuentoVenta"].fillna(0.0).astype(float)

# Calcular descuento por línea y subtotal neto (DescuentoVenta es porcentaje)
detalle_all["Descuento"] = detalle_all["Subtotal"] * detalle_all["DescuentoVenta"]
detalle_all["SubtotalNeto"] = detalle_all["Subtotal"] - detalle_all["Descuento"]

# Crear DetalleID único
detalle_all = detalle_all.reset_index(drop=True)
detalle_all.insert(0, "DetalleID", range(1, len(detalle_all)+1))

# Seleccionar columnas finales con nombres exactos
detalle_facturas = detalle_all[[
    "DetalleID","FacturaID","ProductoID","Cantidad","PrecioUnitario","Subtotal","Descuento","SubtotalNeto"
]].copy()

print("Registros en DetalleFacturas:", len(detalle_facturas))
detalle_facturas.head()


Registros en DetalleFacturas: 90039


,DetalleID,FacturaID,ProductoID,Cantidad,PrecioUnitario,Subtotal,Descuento,SubtotalNeto
0,1,1,1,1,9600000.0,9600000.0,0.0,9600000.0
1,2,2,2,1,4000000.0,4000000.0,0.0,4000000.0
2,3,3,3,2,5600000.0,11200000.0,0.0,11200000.0
3,4,4,4,1,4800000.0,4800000.0,0.0,4800000.0
4,5,5,5,1,5200000.0,5200000.0,0.0,5200000.0


### Validaciones
Revisamos que:
- FacturaID sea único en Facturas.
- Todas las FacturaID en DetalleFacturas existan en Facturas.
- Todos los ProductoID en DetalleFacturas existan en Productos.


In [42]:
errors = []

# 1) FacturaID único en Facturas
if facturas["FacturaID"].isna().any():
    errors.append("Algunas FacturaID son nulas en Facturas.")
if facturas["FacturaID"].duplicated().any():
    dup = facturas[facturas["FacturaID"].duplicated(keep=False)].sort_values("FacturaID")
    errors.append(f"Facturas con FacturaID duplicado: {dup['FacturaID'].unique().tolist()[:10]}")

# 2) FacturaID en DetalleFacturas existe en Facturas
fact_ids = set(facturas["FacturaID"].dropna().astype(int))
detalle_ids = set(detalle_facturas["FacturaID"].dropna().astype(int))
missing_fact = detalle_ids - fact_ids
if missing_fact:
    errors.append(f"DetalleFacturas apunta a FacturaID inexistente: ejemplo {list(missing_fact)[:10]}")

# 3) ProductoID en DetalleFacturas existe en Productos
prod_ids = set(productos_df["ProductoID"])
detalle_prod_missing = set(detalle_facturas[detalle_facturas["ProductoID"].isna()].index)
if len(detalle_prod_missing) > 0:
    errors.append(f"Algunas filas de DetalleFacturas no tienen ProductoID (ej. {list(detalle_prod_missing)[:10]})")

# Mostrar resultados
if not errors:
    print("OK — todas las validaciones pasaron.")
else:
    print("Errores detectados:")
    for e in errors:
        print("-", e)

# Muestras para revisar manualmente
if missing_fact:
    display(detalle_facturas[detalle_facturas["FacturaID"].isin(list(missing_fact))].head(10))
if len(detalle_prod_missing) > 0:
    display(detalle_facturas[detalle_facturas["ProductoID"].isna()].head(10))


Errores detectados:
- Facturas con FacturaID duplicado: [913, 1515, 4362, 6174, 10569, 12203, 13399, 19095, 19606, 21772]


### Reportes rápidos
Total de ventas por marca y top 10 productos más vendidos (por cantidad).


In [43]:
ventas_por_marca = (
    detalle_facturas
    .merge(productos_df[["ProductoID","MarcaProducto"]], on="ProductoID", how="left")
    .groupby("MarcaProducto", dropna=False)["SubtotalNeto"]
    .sum()
    .sort_values(ascending=False)
)

print("Total de ventas por marca:\n", ventas_por_marca.head(10))

top_productos = (
    detalle_facturas
    .merge(productos_df[["ProductoID","NombreProducto"]], on="ProductoID", how="left")
    .groupby("NombreProducto")["Cantidad"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nTop 10 productos más vendidos:\n", top_productos)


Total de ventas por marca:
 MarcaProducto
Lenovo       1.149855e+11
HP           9.445895e+10
Dell         8.463916e+10
Apple        7.607689e+10
Asus         3.239673e+10
Acer         2.146478e+10
Samsung      1.245680e+10
MSI          8.311020e+09
Microsoft    8.017840e+09
Razer        3.838760e+09
Name: SubtotalNeto, dtype: float64

Top 10 productos más vendidos:
 NombreProducto
HP Spectre x360              7730
Lenovo Legion 5 Pro          5864
Lenovo ThinkPad X1 Carbon    5814
Lenovo Yoga 7i               5054
HP Omen 16                   5015
Lenovo IdeaPad 5             4972
Dell XPS 13                  4530
Dell Latitude 7420           4509
HP Pavilion 15               4030
Dell Alienware m15           3658
Name: Cantidad, dtype: int64


### Exportar
Se guardan las tablas listas para importar a Power BI.


In [44]:
with pd.ExcelWriter("modeloVentas.xlsx", engine="openpyxl") as writer:
    ciudades_df.rename(columns={"NombreCiudad":"NombreCiudad"}).to_excel(writer, sheet_name="Ciudades", index=False)
    sucursales_df.to_excel(writer, sheet_name="Sucursales", index=False)
    vendedores_df.to_excel(writer, sheet_name="Vendedores", index=False)
    clientes_df.to_excel(writer, sheet_name="Clientes", index=False)
    productos_df.to_excel(writer, sheet_name="Productos", index=False)
    facturas.to_excel(writer, sheet_name="Facturas", index=False)
    detalle_facturas.to_excel(writer, sheet_name="DetalleFacturas", index=False)

print("Archivo 'modeloVentas.xlsx' generado correctamente.")


Archivo 'modeloVentas.xlsx' generado correctamente.
